# Tiny Dreamer Highway — Colab Optimized Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

Use this notebook to run the **optimized** configuration and compare against the baseline.

## Optimization intent

This notebook should be your comparison point for optimizer-level tuning while keeping the Dreamer-aligned model changes fixed.

Compare runtime, mean reward, crash rate, observation likelihood, and overshooting consistency against the baseline and H100 runs.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet
# Install flashoptim (optional; training falls back to AdamW if unavailable)
python -m pip install flashoptim --quiet || echo 'flashoptim unavailable — using AdamW fallback'

Already up to date.


From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD


In [ ]:
import json
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'optimized_experiment.yaml'
config = load_experiment_config(CONFIG_PATH)

def print_sequence_sampling_guidance(run_label, warm_start_steps):
    effective_warm_start = config.training.warm_start_steps if warm_start_steps is None else warm_start_steps
    nominal_floor = config.training.batch_size * config.replay.sequence_length
    print(f'{run_label} sequence length:', config.replay.sequence_length)
    print(f'{run_label} max episode steps:', config.env.max_episode_steps)
    print(f'{run_label} offroad terminal:', config.env.reward.offroad_terminal)
    print(f'{run_label} effective warm-start steps:', effective_warm_start)
    print(f'{run_label} nominal warm-start floor (batch x sequence):', nominal_floor)
    print('Trainer safeguard: extra random warm-start data will be collected automatically if valid sequences are still unavailable.')
    if config.replay.sequence_length > config.env.max_episode_steps:
        print('Warning: sequence_length exceeds max_episode_steps, so replay sequence sampling is impossible.')
    elif config.env.reward.offroad_terminal and config.replay.sequence_length >= config.env.max_episode_steps - 8:
        print('Caution: sequence_length is close to the episode horizon while offroad_terminal=True; short crash-heavy warm starts may require extra random collection.')
    elif effective_warm_start < nominal_floor:
        print('Caution: warm-start is below the nominal batch_size x sequence_length floor.')

print('Loaded config from:', CONFIG_PATH)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Batch size:', config.training.batch_size)
print('Grad clip norm:', config.training.grad_clip_norm)
print('LR warm-up steps:', config.training.lr_warmup_steps)
print('World-model updates/cycle:', config.training.world_model_updates_per_cycle)
print('Behavior updates/cycle:', config.training.behavior_updates_per_cycle)

Loaded config from: /content/CSC_580_Final_Project/examples/optimized_experiment.yaml
GPU: NVIDIA A100-SXM4-80GB
Batch size: 32
Grad clip norm: 100.0
LR warm-up steps: 50
World-model updates/cycle: 2
Behavior updates/cycle: 2


In [ ]:
RUN_NAME = 'optimized_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CYCLES = None
WARM_START_STEPS = None
POLICY_STEPS = None
CHECKPOINT_INTERVAL = None
RESUME_FROM = None

print('Run name:', RUN_NAME)
print('Effective cycles:', config.training.cycles if CYCLES is None else CYCLES)
print('Effective policy steps:', config.training.policy_steps if POLICY_STEPS is None else POLICY_STEPS)
print_sequence_sampling_guidance('Optimized run', WARM_START_STEPS)

Run name: optimized_run_001
Effective cycles: 500


In [ ]:
print('Launching optimized run. Per-cycle progress lines will appear below.')

training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('Completed cycles:', training_summary.completed_cycles)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Latest metrics:', training_summary.latest_record)

Launching optimized run. Per-cycle progress lines will appear below.
[train] starting run | cycles=500 | start_step=1 | warm_start_steps=1000 | policy_steps=32 | device=cuda


In [ ]:
summary_path = training_summary.log_dir / 'latest_summary.json'
summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
summary_payload

In [ ]:
from IPython.display import Image, display
import csv
import importlib
import json
import tiny_dreamer_highway.evaluation.training_analysis as training_analysis

training_analysis = importlib.reload(training_analysis)
export_training_history_artifacts = training_analysis.export_training_history_artifacts

analysis_outputs = export_training_history_artifacts(
    training_summary.log_dir / 'cycle_metrics.csv',
    RUN_ARTIFACT_ROOT / 'analysis',
    prefix=RUN_NAME,
)

display(Image(filename=str(analysis_outputs['curves'])))
analysis_summary = json.loads(analysis_outputs['summary'].read_text(encoding='utf-8'))
with (training_summary.log_dir / 'cycle_metrics.csv').open('r', encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
latest_metrics_row = metric_rows[-1]
{
    'analysis_summary': analysis_summary,
    'phase4_metrics': {
        'world_model/reconstruction_loss': latest_metrics_row.get('world_model/reconstruction_loss'),
        'world_model/reconstruction_mse': latest_metrics_row.get('world_model/reconstruction_mse'),
        'world_model/observation_log_prob': latest_metrics_row.get('world_model/observation_log_prob'),
        'world_model/overshooting_kl_loss': latest_metrics_row.get('world_model/overshooting_kl_loss'),
        'world_model/overshooting_feature_mse': latest_metrics_row.get('world_model/overshooting_feature_mse'),
        'world_model/overshooting_pairs': latest_metrics_row.get('world_model/overshooting_pairs'),
        'evaluation/mean_reward': latest_metrics_row.get('evaluation/mean_reward'),
        'evaluation/crash_rate': latest_metrics_row.get('evaluation/crash_rate'),
    },
}

## Agent Driving Demo

Record agent-driving GIFs using the optimized trained policy. Compare these visually against the baseline demos.

In [ ]:
from IPython.display import Image, display
import importlib
import tiny_dreamer_highway.evaluation as evaluation_pkg

try:
    evaluation_pkg = importlib.reload(evaluation_pkg)
    record_demo_videos = evaluation_pkg.record_demo_videos
except (AttributeError, ImportError):
    from tiny_dreamer_highway.evaluation.policy_rollout import record_demo_videos

print('Using demo recorder from:', record_demo_videos.__module__)

demo_outputs = record_demo_videos(
    config,
    checkpoint_path=training_summary.latest_checkpoint,
    output_dir=RUN_ARTIFACT_ROOT / 'demo_videos',
    num_episodes=3,
    max_steps=200,
    fps=15,
    seed=config.seed,
    prefix=RUN_NAME,
    device=config.device,
)

for gif_path in demo_outputs.video_paths:
    print(f'\n{gif_path.name}')
    display(Image(filename=str(gif_path)))